# 02 — Quality control and filtering

Equivalent to `scripts/02_qc_filter.py`. Produces **Figure 1** of the report.

Droplet data contains three things that are not cells: empty droplets with ambient
RNA (few genes), dying cells (high mitochondrial fraction), and doublets (two cells
in one droplet, so too many genes). Each distorts clustering differently.

**Look at the distributions before you cut.** Thresholds copied from a tutorial are
not a justification you can defend at grading.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad

import config

sc.settings.verbosity = 3
sc.settings.figdir = config.FIG_DIR
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)
sc.logging.print_header()

In [ ]:
adata = sc.read_h5ad(config.H5AD_RAW)
print(f'{adata.n_obs:,} cells x {adata.n_vars:,} genes')

## Flag mitochondrial and ribosomal genes

*Drosophila* naming is **not** the human convention — there is no `MT-` prefix.
Mito genes are `mt:` (`mt:CoI`, `mt:ND1`, `mt:Cytb`); ribosomal proteins are
`RpS`/`RpL`. Getting this wrong gives every cell 0% mt, you filter nothing, and
nothing warns you.

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('mt:')
adata.var['ribo'] = adata.var_names.str.startswith(('RpS', 'RpL', 'rps', 'rpl'))

print('mitochondrial genes:', int(adata.var['mt'].sum()))
print('ribosomal genes:   ', int(adata.var['ribo'].sum()))

assert adata.var['mt'].sum() > 0, (
    'No mt: genes found — your var_names are probably FlyBase IDs. Redo step 01.')

In [ ]:
sc.pp.calculate_qc_metrics(
    adata, qc_vars=['mt', 'ribo'], percent_top=None, log1p=False, inplace=True
)
adata.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].describe().round(2)

## Look before you cut

In [ ]:
sc.pl.violin(
    adata,
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo'],
    groupby='condition', jitter=0.4, multi_panel=True, rotation=45,
)

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', color='pct_counts_mt')

**Read these plots before continuing.**

- Does the mt% distribution peak below 10? If it peaks above, a 10% cut throws
  away real cells and you should raise it — and say so in Methods.
- Does one sample look different from the others? That is a batch effect to
  remember when you see the UMAP in step 03.
- Where does the cloud of low-gene droplets end? That is your real `min_genes`.

In [ ]:
# Per-sample summary — goes into the Methods section
g = adata.obs.groupby('sample', observed=True)
before = pd.DataFrame({
    'n_cells': g.size(),
    'median_genes': g['n_genes_by_counts'].median(),
    'median_counts': g['total_counts'].median(),
    'median_pct_mt': g['pct_counts_mt'].median(),
}).round(2)
before

## Filter

Thresholds come from `config.py`. Currently following the **paper's** Methods
(300–2500 genes, min 5 cells per gene) rather than the guide's looser numbers,
because RQ1 asks us to reproduce their 36 clusters.

In [ ]:
print('min_genes    :', config.MIN_GENES_PER_CELL)
print('max_genes    :', config.MAX_GENES_PER_CELL)
print('min_cells    :', config.MIN_CELLS_PER_GENE)
print('max_pct_mt   :', config.MAX_PCT_MT)

In [ ]:
n0 = adata.n_obs

sc.pp.filter_cells(adata, min_genes=config.MIN_GENES_PER_CELL)
print(f'after min_genes: {adata.n_obs:,}')

if config.MAX_GENES_PER_CELL is not None:
    adata = adata[adata.obs['n_genes_by_counts'] < config.MAX_GENES_PER_CELL].copy()
    print(f'after max_genes (doublets): {adata.n_obs:,}')

adata = adata[adata.obs['pct_counts_mt'] < config.MAX_PCT_MT].copy()
print(f'after pct_mt: {adata.n_obs:,}')

sc.pp.filter_genes(adata, min_cells=config.MIN_CELLS_PER_GENE)
print(f'genes retained: {adata.n_vars:,}')

print(f'\nRetained {adata.n_obs:,}/{n0:,} cells ({100*adata.n_obs/n0:.1f}%)')
print('Paper analysed 86,224 cells.')

## Confirm the design is still balanced

If one arm lost most of its cells, the DE comparison is compromised and you must
say so in the report rather than present the design as intact.

In [ ]:
adata.obs['condition'].value_counts().sort_index()

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             groupby='condition', jitter=0.4, multi_panel=True, rotation=45)

In [ ]:
adata.write(config.H5AD_QC)
print('Wrote', config.H5AD_QC)